# Titanic Dataset - Data Dictionary and Data Cleaning

This notebook documents the Titanic dataset, performs duplicate detection, handles missing values, and exports a cleaned version of the dataset.


## Dictionary

In [2]:
import pandas as pd

df_dicionario = pd.read_csv('dictionary.csv')

pd.set_option('display.max_colwidth', None)

display(df_dicionario)

,Name,Description (Semantic Meaning),Data Type,Units,Possible Values / Categories,Potential Data Quality Issues
0,PassengerId,Unique identifier for each passenger,Numerical,-,Positive integers,Should be unique; possible duplicates if dataset is merged
1,Survived,Indicates if the passenger survived,Numerical (binary),-,0 = No; 1 = Yes,Class imbalance (more non-survivors)
2,Pclass,Passenger class (socio-economic status),Numerical (categorical),-,1 = 1st; 2 = 2nd; 3 = 3rd,Ordinal meaning may be misinterpreted as continuous
3,Name,Full name of the passenger,Text,-,Free text,Long strings; titles included (Mr Mrs etc); needs parsing
4,Sex,Gender of the passenger,Categorical,-,male; female,No major issues but must be encoded
5,Age,Age of the passenger,Numerical,Years,> 0,Missing values; possible outliers
6,SibSp,Number of siblings/spouses aboard,Numerical,Count,>= 0,Skewed distribution (many zeros)
7,Parch,Number of parents/children aboard,Numerical,Count,>= 0,Skewed distribution
8,Ticket,Ticket number,Text,-,Alphanumeric,Inconsistent formats; duplicates possible
9,Fare,Ticket fare paid,Numerical,Currency (£),>= 0,Outliers; skewed distribution


## Load Titanic Dataset

In [3]:
import pandas as pd

df = pd.read_csv('Titanic-Dataset.csv')
print(f"Dataset shape before cleaning: {df.shape}")

Dataset shape before cleaning: (891, 12)


## Duplicate Detection and Handling

**Detection Strategy:**  
To identify duplicates, the full dataset was analyzed to detect identical rows. Since `PassengerId` is a unique identifier, it was also considered to verify uniqueness.

**Consolidation Method:**  
No duplicate rows were found, so no removal was necessary.


In [4]:
# Identifying exact row duplicates
exact_duplicates = df.duplicated().sum()
print(f"Exact row duplicates: {exact_duplicates}")

# Verifying PassengerId uniqueness
passenger_id_duplicates = df.duplicated(subset=['PassengerId']).sum()
print(f"Duplicates by PassengerId: {passenger_id_duplicates}")

Exact row duplicates: 0
Duplicates by PassengerId: 0


## Missing Data Treatment

In [5]:
# Missing values per variable
missing_per_variable = df.isnull().sum()
print("Missing values per variable:")
print(missing_per_variable[missing_per_variable > 0])

# Missing values overall
total_missing = df.isnull().sum().sum()
print(f"\nTotal missing values in the dataset: {total_missing}")

Missing values per variable:
Age         177
Cabin       687
Embarked      2
dtype: int64

Total missing values in the dataset: 866


**Data Missing Patterns:**  
- `Age`: can be treated as MCAR/MAR, since some ages were not recorded.  
- `Cabin`: has a high percentage of missing values, likely because many cabin assignments were not recorded.  
- `Embarked`: has very few missing values and can be treated safely with imputation.

**Treatment Strategy and Justification:**  
- `Age`: filled with the median value, since median is robust to outliers.  
- `Embarked`: filled with the most frequent value (mode).  
- `Cabin`: removed due to the high percentage of missing values.

This strategy preserves most of the dataset while removing the least useful incomplete information.


In [6]:
df_cleaned = df.copy()

# Treating missing values
df_cleaned['Age'] = df_cleaned['Age'].fillna(df_cleaned['Age'].median())
df_cleaned['Embarked'] = df_cleaned['Embarked'].fillna(df_cleaned['Embarked'].mode()[0])

# Removing Cabin due to excessive missing values
df_cleaned = df_cleaned.drop(columns=['Cabin'])

print(f"Total missing after treatment: {df_cleaned.isnull().sum().sum()}")
print(f"\nDataset after cleaning: {df_cleaned.shape}")

df_cleaned.to_csv('titanic_cleaned.csv', index=False)

Total missing after treatment: 0

Dataset after cleaning: (891, 11)


## Final Statement

The dataset was preprocessed by checking for duplicates, handling missing values through median imputation for `Age`, mode imputation for `Embarked`, and removal of the `Cabin` column due to excessive missing data. As a result, the final dataset is clean and consistent for future analysis.
